## Import and S3 paths

In [25]:
import pandas as pd
from sklearn.model_selection import train_test_split

# S3 paths
BUCKET = "prognostica-cancer-project"
S3_RAW = f"s3://{BUCKET}/raw/"
S3_PROCESSED = f"s3://{BUCKET}/processed/"
S3_FEATURES = f"s3://{BUCKET}/features/"

## Load datasets from S3

In [26]:
# Raw datasets
lung = pd.read_csv(S3_RAW + "lcs_synthetic_20000.csv")
cdc = pd.read_csv(S3_RAW + "US_cancer_incidence_1999_2022.csv")

teq_2020 = pd.read_csv(S3_RAW + "TEQ_2020.csv")
teq_2021 = pd.read_csv(S3_RAW + "TEQ_2021.csv")
teq_2022 = pd.read_csv(S3_RAW + "TEQ_2022.csv")
teq_2023 = pd.read_csv(S3_RAW + "TEQ_2023.csv")
teq_2024 = pd.read_csv(S3_RAW + "TEQ_2024.csv")

In [27]:
print("Lung shape:", lung.shape)
print("CDC shape:", cdc.shape)
print("TEQ 2020 shape:", teq_2020.shape)
print("TEQ 2021 shape:", teq_2021.shape)
print("TEQ 2022 shape:", teq_2022.shape)
print("TEQ 2023 shape:", teq_2023.shape)
print("TEQ 2024 shape:", teq_2024.shape)

Lung shape: (20000, 16)
CDC shape: (60, 4)
TEQ 2020 shape: (818, 90)
TEQ 2021 shape: (812, 90)
TEQ 2022 shape: (813, 90)
TEQ 2023 shape: (770, 90)
TEQ 2024 shape: (721, 90)


In [28]:
print("Lung columns:")
print(lung.columns)

print("\nCDC columns:")
print(cdc.columns)

print("\nTEQ columns:")
print(teq_2020.columns)

Lung columns:
Index(['GENDER', 'AGE', 'SMOKING', 'YELLOW_FINGERS', 'ANXIETY',
       'PEER_PRESSURE', 'CHRONIC DISEASE', 'FATIGUE ', 'ALLERGY ', 'WHEEZING',
       'ALCOHOL CONSUMING', 'COUGHING', 'SHORTNESS OF BREATH',
       'SWALLOWING DIFFICULTY', 'CHEST PAIN', 'LUNG_CANCER'],
      dtype='object')

CDC columns:
Index(['Notes', 'Leading Cancer Sites', 'Leading Cancer Sites Code', 'Count'], dtype='object')

TEQ columns:
Index(['Year', 'TRI Facility ID', 'Facility Name', 'Street Address', 'City',
       'County', 'ST', 'ZIP', 'Latitude', 'Longitude', 'Primary NAICS',
       'NAICS 2', 'NAICS 3', 'NAICS 4', 'NAICS 5', 'NAICS 6',
       'Parent Company Name', 'Parent Company DB Number', 'Doc_Ctrl_Num',
       'Chemical', 'CAS #/Compound ID', 'Congener Number', 'Congener CAS#',
       'Congener', 'Clean Air Act Chemical', 'Classification', 'Metal',
       'Metal Category', 'Carcinogen', 'Form Type', 'Unit of Measure',
       '5.1 Fugitive Air', '5.2 Stack Air', '5.3 Water',
       '5.4.

## Combine the TEQ files

In [29]:
teq_all = pd.concat(
    [teq_2020, teq_2021, teq_2022, teq_2023, teq_2024],
    ignore_index=True
)

print("Combined TEQ shape:", teq_all.shape)

Combined TEQ shape: (3934, 90)


## Clean TEQ column names and remove junk columns

In [30]:
teq_all.columns = (
    teq_all.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace(".", "", regex=False)
    .str.replace("-", "_")
    .str.replace("/", "_")
    .str.replace("#", "number")
)

teq_all = teq_all.drop(columns=["20251105", "klj"], errors="ignore")

print(teq_all.columns)

Index(['year', 'tri_facility_id', 'facility_name', 'street_address', 'city',
       'county', 'st', 'zip', 'latitude', 'longitude', 'primary_naics',
       'naics_2', 'naics_3', 'naics_4', 'naics_5', 'naics_6',
       'parent_company_name', 'parent_company_db_number', 'doc_ctrl_num',
       'chemical', 'cas_number_compound_id', 'congener_number',
       'congener_casnumber', 'congener', 'clean_air_act_chemical',
       'classification', 'metal', 'metal_category', 'carcinogen', 'form_type',
       'unit_of_measure', '51_fugitive_air', '52_stack_air', '53_water',
       '541_underground_class_i', '542_underground_class_ii_v',
       '551a_rcra_c_landfills', '551b_other_landfills', '552_land_treatment',
       '553a_rcra_c_surface_impoundment', '553b_other_surface_impoundment',
       '554_other_disposal', 'on_site_release_total', '61_potw', '62_m10',
       '62_m41', '62_m62', '62_m81', '62_m82', '62_m66', '62_m67', '62_m64',
       '62_m65', '62_m73', '62_m79', '62_m90', '62_m94', '62_m

## Keep and save TEQ subset for analysis

In [31]:
teq_keep = [
    "year",
    "tri_facility_id",
    "facility_name",
    "city",
    "county",
    "st",
    "latitude",
    "longitude",
    "chemical",
    "carcinogen",
    "unit_of_measure",
    "on_site_release_total",
    "off_site_release_total",
    "total_releases"
]

teq_clean = teq_all[[c for c in teq_keep if c in teq_all.columns]].copy()

print(teq_clean.shape)
teq_clean.head()

(3934, 14)


,year,tri_facility_id,facility_name,city,county,st,latitude,longitude,chemical,carcinogen,unit_of_measure,on_site_release_total,off_site_release_total,total_releases
0,2020,00785SPRTRKM142,AES PUERTO RICO LP,GUAYAMA,GUAYAMA MUNICIPIO,PR,17.943889,-66.150917,Dioxin and dioxin-like compounds,YES,Grams,0.082869,0.00000,0
1,2020,00804VRGNSKRUMB,VIRGIN ISLANDS WATER & POWER AUTHORITY,SAINT THOMAS,ST. THOMAS ISLAND,VI,18.330234,-64.960147,Dioxin and dioxin-like compounds,YES,Grams,0.004048,0.00000,0
2,2020,00821VRGNSESTAT,VIRGIN ISLANDS WATER & POWER AUTHORITY,CHRISTIANSTED,ST. CROIX ISLAND,VI,17.750141,-64.714793,Dioxin and dioxin-like compounds,YES,Grams,0.000661,0.00000,0
3,2020,00936SNJNCKM267,ARGOS PUERTO RICO CORP.,DORADO,DORADO MUNICIPIO,PR,18.394400,-66.297600,Dioxin and dioxin-like compounds,YES,Grams,0.011381,0.00000,0
4,2020,04239NTRNTRILEY,PIXELLE SPECIALTY SOLUTIONS,JAY,FRANKLIN,ME,44.507100,-70.238810,Dioxin and dioxin-like compounds,YES,Grams,0.018633,0.01163,0


In [32]:
teq_clean.to_csv(S3_PROCESSED + "teq_combined_cleaned.csv", index=False)

## Clean & save CDC dataset

In [33]:
cdc.columns = (
    cdc.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

print(cdc.columns)
cdc.head()

Index(['notes', 'leading_cancer_sites', 'leading_cancer_sites_code', 'count'], dtype='object')


,notes,leading_cancer_sites,leading_cancer_sites_code,count
0,NaN,Brain and Other Nervous System,31010-31040,523196.0
1,NaN,Breast,26000,5558064.0
2,NaN,Cervix Uteri,27010,310804.0
3,NaN,Colon and Rectum,21041-21052,3521960.0
4,NaN,Corpus Uteri,27020,1121662.0


In [34]:
cdc.to_csv(S3_PROCESSED + "cdc_incidence_cleaned.csv", index=False)

# Clean lung dataset

In [35]:
lung.columns = (
    lung.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

print(lung.columns)
lung.head()

Index(['gender', 'age', 'smoking', 'yellow_fingers', 'anxiety',
       'peer_pressure', 'chronic_disease', 'fatigue', 'allergy', 'wheezing',
       'alcohol_consuming', 'coughing', 'shortness_of_breath',
       'swallowing_difficulty', 'chest_pain', 'lung_cancer'],
      dtype='object')


,gender,age,smoking,yellow_fingers,anxiety,peer_pressure,chronic_disease,fatigue,allergy,wheezing,alcohol_consuming,coughing,shortness_of_breath,swallowing_difficulty,chest_pain,lung_cancer
0,M,69,2,1,1,2,1,2,1,1,2,2,2,1,1,YES
1,M,71,2,2,1,1,2,1,2,2,1,1,2,2,1,YES
2,M,61,2,1,1,2,2,1,2,2,1,1,2,2,2,NO
3,M,55,2,2,1,2,1,1,1,2,2,1,2,2,2,YES
4,F,56,2,1,1,1,1,2,2,2,2,1,2,2,2,YES


## drop duplicates

In [36]:
lung = lung.drop_duplicates()
print("Lung shape after duplicates removed:", lung.shape)

Lung shape after duplicates removed: (19571, 16)


## check missing values

In [37]:
print(lung.isnull().sum())

gender                   0
age                      0
smoking                  0
yellow_fingers           0
anxiety                  0
peer_pressure            0
chronic_disease          0
fatigue                  0
allergy                  0
wheezing                 0
alcohol_consuming        0
coughing                 0
shortness_of_breath      0
swallowing_difficulty    0
chest_pain               0
lung_cancer              0
dtype: int64


# Verify target & gender columns

In [38]:
print(lung["gender"].value_counts(dropna=False))
print(lung["lung_cancer"].value_counts(dropna=False))

gender
M    10322
F     9249
Name: count, dtype: int64
lung_cancer
YES    17012
NO      2559
Name: count, dtype: int64


# Feature engineering on lung dataset

In [39]:
lung["age_group"] = pd.cut(
    lung["age"],
    bins=[0, 40, 50, 60, 70, 120],
    labels=["<=40", "41-50", "51-60", "61-70", "71+"]
)

## one hot encode age groups

In [40]:
lung_model = pd.get_dummies(lung, columns=["age_group"], drop_first=True)

print(lung_model.shape)
lung_model.head()

(19571, 20)


,gender,age,smoking,yellow_fingers,anxiety,peer_pressure,chronic_disease,fatigue,allergy,wheezing,alcohol_consuming,coughing,shortness_of_breath,swallowing_difficulty,chest_pain,lung_cancer,age_group_41-50,age_group_51-60,age_group_61-70,age_group_71+
0,M,69,2,1,1,2,1,2,1,1,2,2,2,1,1,YES,False,False,True,False
1,M,71,2,2,1,1,2,1,2,2,1,1,2,2,1,YES,False,False,False,True
2,M,61,2,1,1,2,2,1,2,2,1,1,2,2,2,NO,False,False,True,False
3,M,55,2,2,1,2,1,1,1,2,2,1,2,2,2,YES,False,True,False,False
4,F,56,2,1,1,1,1,2,2,2,2,1,2,2,2,YES,False,True,False,False


# Check class imbalance

In [41]:
print(lung_model["lung_cancer"].value_counts())
print(lung_model["lung_cancer"].value_counts(normalize=True))

lung_cancer
YES    17012
NO      2559
Name: count, dtype: int64
lung_cancer
YES    0.869245
NO     0.130755
Name: proportion, dtype: float64


The lung cancer target variable is imbalanced, with substantially more positive cases than negative cases. This will be addressed during model training through stratified splitting and, if needed, class weighting or resampling.

# Split into train, validation, and test sets

In [42]:
train_df, temp_df = train_test_split(
    lung_model,
    test_size=0.30,
    random_state=42,
    stratify=lung_model["lung_cancer"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["lung_cancer"]
)

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)

Train shape: (13699, 20)
Validation shape: (2936, 20)
Test shape: (2936, 20)


# Save model-ready files to S3

In [43]:
lung_model.to_csv(S3_FEATURES + "lung_model_data.csv", index=False)
train_df.to_csv(S3_FEATURES + "lung_train.csv", index=False)
val_df.to_csv(S3_FEATURES + "lung_val.csv", index=False)
test_df.to_csv(S3_FEATURES + "lung_test.csv", index=False)

print("All cleaned and split files saved to S3.")

All cleaned and split files saved to S3.


The Kaggle lung cancer dataset was selected as the primary supervised learning dataset because it contains row-level predictor variables and a target label suitable for classification. The EPA TEQ files were combined into a single cleaned environmental dataset for supporting exploratory and contextual analysis. The CDC dataset was cleaned and retained for descriptive market insight, but the uploaded version does not currently support direct row-level merging with the lung model dataset. Model-ready lung training, validation, and test files were saved to the features/ folder in S3 for the training phase.